In [1]:
!unzip -q FER2013.zip

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

# --- 1. CONFIGURATION ---
# Ensure your folder structure is: /content/train/[emotion_name]/img.jpg
TRAIN_PATH = '/content/train'
TEST_PATH  = '/content/test'
IMG_HEIGHT, IMG_WIDTH = 48, 48
BATCH_SIZE = 64

# --- 2. UNIFIED DATA PIPELINE ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_dataset = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale', class_mode='categorical',
    subset='training', batch_size=BATCH_SIZE, shuffle=True
)

valid_dataset = valid_datagen.flow_from_directory(
    TRAIN_PATH, target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale', class_mode='categorical',
    subset='validation', batch_size=BATCH_SIZE, shuffle=True
)

# --- 3. CLASS WEIGHTS (Fix for single-class prediction) ---
# This calculates weights to penalize the model for missing rare emotions
y_train_labels = train_dataset.classes
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weights_dict = dict(enumerate(class_weights))

# --- 4. MODEL ARCHITECTURE ---
model = Sequential([
    Conv2D(64, (3,3), padding='same', input_shape=(48, 48, 1)),
    BatchNormalization(), Activation('relu'), MaxPooling2D(2, 2),

    Conv2D(128, (5,5), padding='same'),
    BatchNormalization(), Activation('relu'), MaxPooling2D(2, 2),

    Conv2D(512, (3,3), padding='same'),
    BatchNormalization(), Activation('relu'), MaxPooling2D(2, 2),

    Flatten(),
    Dense(256), BatchNormalization(), Activation('relu'), Dropout(0.5),
    Dense(512), BatchNormalization(), Activation('relu'), Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# --- 5. CALLBACKS & TRAINING ---
lrd = ReduceLROnPlateau(monitor='val_loss', patience=5, verbose=1, factor=0.5, min_lr=1e-7)
mcp = ModelCheckpoint('best_model.keras', save_best_only=True)
es = EarlyStopping(monitor='val_loss', patience=15, verbose=1)

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=50,
    class_weight=class_weights_dict, # Key fix included here
    callbacks=[lrd, mcp, es]
)

Found 22968 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 51s 110ms/step - accuracy: 0.1925 - loss: 2.3320 - val_accuracy: 0.1893 - val_loss: 2.0434 - learning_rate: 1.0000e-04
Epoch 2/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.2016 - loss: 2.1974 - val_accuracy: 0.2916 - val_loss: 1.7789 - learning_rate: 1.0000e-04
Epoch 3/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.2186 - loss: 2.1150 - val_accuracy: 0.3318 - val_loss: 1.7089 - learning_rate: 1.0000e-04
Epoch 4/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 23s 65ms/step - accuracy: 0.2368 - loss: 2.0405 - val_accuracy: 0.3120 - val_loss: 1.8048 - learning_rate: 1.0000e-04
Epoch 5/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 23s 64ms/step - accuracy: 0.2620 - loss: 1.9441 - val_accuracy: 0.2904 - val_loss: 1.9202 - learning_rate: 1.0000e-04
Epoch 6/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.2714 - loss: 1.8883 - val_accuracy: 0.3461 - val_loss: 1.6918 - learning_rate: 1.0000e-04
Epoch 7/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 25s 70ms/st

In [18]:
# 1. Ensure the test generator is configured for evaluation
# Important: shuffle must be False for accurate comparison with y_true
test_datagen = ImageDataGenerator(rescale=1./255)
test_dataset = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # CRITICAL: Do not shuffle for evaluation
)

# 2. Predict
# Resetting the generator is a good safety habit
test_dataset.reset()
Y_pred = model.predict(test_dataset, steps=len(test_dataset))
y_pred = np.argmax(Y_pred, axis=1)

# 3. Get True Labels
# test_dataset.classes are available because shuffle=False
y_true = test_dataset.classes

# 4. Print Results
from sklearn.metrics import classification_report, confusion_matrix
target_names = list(test_dataset.class_indices.keys())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

Found 7178 images belonging to 7 classes.
113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step
Confusion Matrix:
[[ 496   86   34   50  189   72   31]
 [  18   83    1    0    5    1    3]
 [ 163   51  212   38  221  176  163]
 [  73   10   24 1414  165   29   59]
 [  87   19   31   71  899  102   24]
 [ 189   51   68   86  407  415   31]
 [  35   15   34   37   46    9  655]]

Classification Report:
              precision    recall  f1-score   support

       angry       0.47      0.52      0.49       958
     disgust       0.26      0.75      0.39       111
        fear       0.52      0.21      0.30      1024
       happy       0.83      0.80      0.81      1774
     neutral       0.47      0.73      0.57      1233
         sad       0.52      0.33      0.40      1247
    surprise       0.68      0.79      0.73       831

    accuracy                           0.58      7178
   macro avg       0.54      0.59      0.53      7178
weighted avg       0.60      0.58      0.57      7178



In [20]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# 1. Load the model
# Ensure 'model.keras' is in the same folder as this script
model = tf.keras.models.load_model('best_model.keras', compile=False)

# Match these labels EXACTLY to your training folder names in alphabetical order
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

def predict_emotion(img):
    if img is None:
        return None

    # Convert input to PIL
    img = Image.fromarray(img.astype('uint8'))

    # Pre-processing steps to match your training (48x48, Grayscale)
    img = img.convert('L')             # Convert to Grayscale
    img = img.resize((48, 48))         # Resize
    img = np.array(img) / 255.0        # Normalize

    # Reshape to (1, 48, 48, 1)
    img_input = img.reshape(1, 48, 48, 1)

    # Predict
    predictions = model.predict(img_input)

    # Return as a dictionary for Gradio Label component
    return {emotion_labels[i]: float(predictions[0][i]) for i in range(7)}

# 2. Launch Interface
iface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(), # Removed type="pil" to let Gradio handle it naturally
    outputs=gr.Label(num_top_classes=7),
    title="Facial Emotion Recognition",
    description="Upload an image to detect the emotion."
)

if __name__ == "__main__":
    iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a3339b5513975876ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7865 <> https://a3339b5513975876ad.gradio.live
